In [2]:
import os
from pathlib import Path

# Get the directory of the notebook, then its parent (the project root)
PROJECT_ROOT = Path(os.getcwd()).parent
college_stats_dir = PROJECT_ROOT / "data" / "college_stats" / "cfbd"
recruits_dir = PROJECT_ROOT / "data" / "Football Recruitment Tables" / "cfbd_recruits"

print(f"Project root directory: {PROJECT_ROOT}")
print(f"College stats directory: {college_stats_dir}")
print(f"Football Recruits directory: {recruits_dir}")

# Ensure existence
if not college_stats_dir.exists():
    print(f"Creating directory: {college_stats_dir}")
    college_stats_dir.mkdir(parents=True, exist_ok=True)
else:
    print("College stats directory exists.")

if not recruits_dir.exists():
    print(f"Creating directory: {recruits_dir}")
    recruits_dir.mkdir(parents=True, exist_ok=True)
else:
    print("Football Recruits directory exists.")


Project root directory: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence
College stats directory: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\college_stats\cfbd
Football Recruits directory: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\Football Recruitment Tables\cfbd_recruits
College stats directory exists.
Football Recruits directory exists.


In [3]:
import time
import cfbd
from cfbd.rest import ApiException
from pprint import pprint
from dotenv import load_dotenv, find_dotenv
import pandas as pd
# Search for the .env file and load it
load_dotenv(find_dotenv("GEMINI_API_KEY.env"))

# Configure the host and authorization
configuration = cfbd.Configuration()
configuration.host = "https://api.collegefootballdata.com"

# The BEARER_TOKEN is used as the access_token for authorization
configuration.access_token = os.environ.get("BEARER_TOKEN")


In [ ]:
# Enter a context with an instance of the API client
with cfbd.ApiClient(configuration) as api_client:
    # Create an instance of the API class
    api_instance = cfbd.GamesApi(api_client)
    
    try:
        # Fetching games for 2025 as requested
        print("Fetching games for 2025...")
        games = api_instance.get_games(year=2025)
        print(f"Successfully fetched {len(games)} games.")
        if games:
            pprint(games[0])
    except ApiException as e:
        print("Exception when calling GamesApi->get_games: %s\n" % e)

games_df = pd.DataFrame([game.to_dict() for game in games])

games_df.head()

,id,season,week,seasonType,startDate,startTimeTBD,completed,neutralSite,conferenceGame,venueId,...,awayClassification,awayPoints,awayLineScores,awayPostgameWinProbability,awayPregameElo,awayPostgameElo,excitementIndex,highlights,notes,attendance
0,401756846,2025,1,SeasonType.REGULAR,2025-08-23 16:00:00+00:00,False,True,True,True,3504.0,...,DivisionClassification.FBS,24.0,"[7, 0, 7, 10]",0.798456,1616.0,1619.0,6.029094,,Aer Lingus College Football Classic,None
1,401767476,2025,1,SeasonType.REGULAR,2025-08-23 17:00:00+00:00,False,True,False,True,3779.0,...,DivisionClassification.FCS,6.0,"[3, 0, 3, 0]",0.290214,NaN,NaN,5.126499,,None,None
2,401760371,2025,1,SeasonType.REGULAR,2025-08-23 20:00:00+00:00,False,True,False,False,6501.0,...,DivisionClassification.FCS,31.0,"[10, 7, 7, 7]",0.021213,NaN,NaN,9.598562,,None,None
3,401767126,2025,1,SeasonType.REGULAR,2025-08-23 20:30:00+00:00,False,True,False,False,4765.0,...,DivisionClassification.FCS,42.0,"[7, 7, 14, 14]",0.971885,NaN,NaN,4.943514,,None,None
4,401756847,2025,1,SeasonType.REGULAR,2025-08-23 22:30:00+00:00,False,True,False,False,3833.0,...,DivisionClassification.FBS,7.0,"[7, 0, 0, 0]",0.000000,1490.0,1441.0,3.783041,,None,None


In [37]:
# ---------------------------------------------------------
# 1. RECRUITING SUMMARIES (2015 - 2028)
# ---------------------------------------------------------
all_recruit_data = []
years_to_fetch = range(2015, 2029)  # 2015 through 2028

with cfbd.ApiClient(configuration) as api_client:
    recruit_api = cfbd.RecruitingApi(api_client)
    
    for year in years_to_fetch:
        print(f"🏈 Fetching recruits for {year}...", end=" ")
        try:
            recruits = recruit_api.get_recruits(year=year)
            
            # Parse CFBD objects into a list of dictionaries with full metadata
            year_data = []
            for r in recruits:
                # Extract nested hometown info if available
                hometown = getattr(r, 'hometown_info', None)
                
                year_data.append({
                    'id': getattr(r, 'id', None),
                    'athlete_id': getattr(r, 'athlete_id', None),
                    'recruit_type': getattr(r, 'recruit_type', None),
                    'year': r.year,
                    'ranking': getattr(r, 'ranking', None),
                    'name': r.name,
                    'high_school': getattr(r, 'school', None),
                    'committed_to': getattr(r, 'committed_to', None),
                    'position': r.position,
                    'height': r.height,
                    'weight': r.weight,
                    'stars': r.stars,
                    'rating': r.rating,
                    'city': getattr(r, 'city', None),
                    'state': r.state_province,
                    'country': getattr(r, 'country', None),
                    # Nested coordinates/FIPS
                    'latitude': getattr(hometown, 'latitude', None) if hometown else None,
                    'longitude': getattr(hometown, 'longitude', None) if hometown else None,
                    'fips_code': getattr(hometown, 'fips_code', None) if hometown else None
                })
            
            all_recruit_data.extend(year_data)
            print(f"✅ Fetched {len(year_data)} recruits.")
            time.sleep(0.5) # Rate limit protection
            
        except ApiException as e:
            print(f"⚠️ Error fetching {year}: {e}")

# Convert to DataFrame and Save
if all_recruit_data:
    df_recruits = pd.DataFrame(all_recruit_data)
    
    # Save the expanded metadata file
    output_path = recruits_dir / f"recruiting_summaries_{years_to_fetch.start}_{years_to_fetch.stop-1}.csv"
    df_recruits.to_csv(output_path, index=False, encoding='utf-8-sig')
    
    print(f"\n🎉 Successfully saved {len(df_recruits)} recruits to: {output_path}")
    display(df_recruits.head())

🏈 Fetching recruits for 2015... ✅ Fetched 3516 recruits.
🏈 Fetching recruits for 2016... ✅ Fetched 3937 recruits.
🏈 Fetching recruits for 2017... ✅ Fetched 4222 recruits.
🏈 Fetching recruits for 2018... ✅ Fetched 3886 recruits.
🏈 Fetching recruits for 2019... ✅ Fetched 3992 recruits.
🏈 Fetching recruits for 2020... ✅ Fetched 3870 recruits.
🏈 Fetching recruits for 2021... ✅ Fetched 2666 recruits.
🏈 Fetching recruits for 2022... ✅ Fetched 2198 recruits.
🏈 Fetching recruits for 2023... ✅ Fetched 2297 recruits.
🏈 Fetching recruits for 2024... ✅ Fetched 2548 recruits.
🏈 Fetching recruits for 2025... ✅ Fetched 2507 recruits.
🏈 Fetching recruits for 2026... ✅ Fetched 3107 recruits.
🏈 Fetching recruits for 2027... ✅ Fetched 0 recruits.
🏈 Fetching recruits for 2028... ✅ Fetched 0 recruits.

🎉 Successfully saved 38746 recruits to: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\Football Recruitment Tables\cfbd_recruits\recruiting_summaries_2015_2028.csv


,id,athlete_id,recruit_type,year,ranking,name,high_school,committed_to,position,height,weight,stars,rating,city,state,country,latitude,longitude,fips_code
0,138315,3915192,RecruitClassification.HIGHSCHOOL,2015,1,Trenton Thompson,Westover,Georgia,DT,74.5,313.0,5,0.9991,Albany,GA,USA,31.578206,-84.155681,13095
1,138316,-1009710,RecruitClassification.HIGHSCHOOL,2015,2,Martez Ivey,Apopka,Florida,OT,77.5,275.0,5,0.9991,Apopka,FL,USA,28.677968,-81.511521,12095
2,138317,None,RecruitClassification.HIGHSCHOOL,2015,3,Byron Cowart,Armwood,Auburn,SDE,76.0,250.0,5,0.9987,Seffner,FL,USA,27.998541,-82.274884,12057
3,138318,3912545,RecruitClassification.HIGHSCHOOL,2015,4,Iman Marshall,Long Beach Poly,USC,CB,73.0,190.0,5,0.9985,Long Beach,CA,USA,33.769016,-118.191605,06037
4,138319,3691739,RecruitClassification.HIGHSCHOOL,2015,5,Derwin James,Haines City,Florida State,S,74.0,201.0,5,0.9982,Auburndale,FL,USA,28.107088,-81.803580,12105


In [ ]:
# Enter a context with an instance of the API client
with cfbd.ApiClient(configuration) as api_client:
    # Create an instance of the API class
    api_instance = cfbd.StatsApi(api_client)

    try:
        api_response = api_instance.get_categories()
        print("The response of StatsApi->get_categories:\n")
        pprint(api_response)
    except Exception as e:
        print("Exception when calling StatsApi->get_categories: %s\n" % e)

The response of StatsApi->get_categories:

['completionAttempts',
 'defensiveTDs',
 'extraPoints',
 'fieldGoalPct',
 'fieldGoals',
 'firstDowns',
 'fourthDownEff',
 'fumblesLost',
 'fumblesRecovered',
 'interceptions',
 'interceptionTDs',
 'interceptionYards',
 'kickingPoints',
 'kickReturns',
 'kickReturnTDs',
 'kickReturnYards',
 'netPassingYards',
 'passesDeflected',
 'passesIntercepted',
 'passingTDs',
 'possessionTime',
 'puntReturns',
 'puntReturnTDs',
 'puntReturnYards',
 'qbHurries',
 'rushingAttempts',
 'rushingTDs',
 'rushingYards',
 'sacks',
 'tackles',
 'tacklesForLoss',
 'thirdDownEff',
 'totalFumbles',
 'totalPenaltiesYards',
 'totalYards',
 'turnovers',
 'yardsPerPass',
 'yardsPerRushAttempt']


In [ ]:
# Enter a context with an instance of the API client
with cfbd.ApiClient(configuration) as api_client:
    # Create an instance of the API class
    api_instance = cfbd.TeamsApi(api_client)
    team = 'Texas' # str | Optional team filter (optional)
    year = 2024 # int | Optional year filter, defaults to 2025 (optional)
    try:
        api_response = api_instance.get_roster(team=team, year=year)
        print("The response of TeamsApi->get_roster:\n")
        pprint(api_response)
    except Exception as e:
        print("Exception when calling TeamsApi->get_roster: %s\n" % e)

The response of TeamsApi->get_roster:

[RosterPlayer(id='4360463', first_name='Jermayne', last_name='Lole', team='Texas', height=75, weight=315, jersey=99, year=4, position='DL', home_city='Long Beach', home_state='CA', home_country='USA', home_latitude=33.7690164, home_longitude=-118.1916048, home_county_fips='06037', recruit_ids=['126820']),
 RosterPlayer(id='4426398', first_name='David', last_name='Gbenda', team='Texas', height=72, weight=235, jersey=33, year=4, position='LB', home_city='Katy', home_state='TX', home_country='USA', home_latitude=29.7857853, home_longitude=-95.8243956, home_county_fips='48157', recruit_ids=['122445']),
 RosterPlayer(id='4426401', first_name='Bill', last_name='Norton', team='Texas', height=78, weight=335, jersey=15, year=4, position='DL', home_city='Memphis', home_state='TN', home_country='USA', home_latitude=35.1490215, home_longitude=-90.0516285, home_county_fips='47157', recruit_ids=['122439']),
 RosterPlayer(id='4427251', first_name='Velton', last_

In [29]:
# 🧪 Inspecting API response structures for reliable DataFrame creation
with cfbd.ApiClient(configuration) as api_client:
    stats_api = cfbd.StatsApi(api_client)
    teams_api = cfbd.TeamsApi(api_client)
    test_year = 2024
    test_team = "Texas"

    print("--- 1. Advanced Season Stats Keys ---")
    res1 = stats_api.get_advanced_season_stats(year=test_year, team=test_team)
    if res1: print(res1[0].to_dict().keys())

    print("\n--- 2. Player Season Stats Keys (Suspected error source) ---")
    res2 = stats_api.get_player_season_stats(year=test_year, team=test_team)
    if res2: 
        d2 = res2[0].to_dict()
        print(f"Keys: {list(d2.keys())}")
        print(f"Sample Row: {d2}")

    print("\n--- 3. Team Stats Keys ---")
    res3 = stats_api.get_team_stats(year=test_year)
    if res3: print(res3[0].to_dict().keys())

    print("\n--- 4. FBS Teams Keys ---")
    res4 = teams_api.get_fbs_teams(year=test_year)
    if res4: print(res4[0].to_dict().keys())

    print("\n--- 5. Roster Keys ---")
    res5 = teams_api.get_roster(year=test_year, team=test_team)
    if res5: print(res5[0].to_dict().keys())


--- 1. Advanced Season Stats Keys ---
dict_keys(['season', 'team', 'conference', 'offense', 'defense'])

--- 2. Player Season Stats Keys (Suspected error source) ---
Keys: ['season', 'playerId', 'player', 'position', 'team', 'conference', 'category', 'statType', 'stat']
Sample Row: {'season': 2024, 'playerId': '4360463', 'player': 'Jermayne Lole', 'position': 'DL', 'team': 'Texas', 'conference': 'SEC', 'category': 'defensive', 'statType': 'PD', 'stat': '0'}

--- 3. Team Stats Keys ---
dict_keys(['season', 'team', 'conference', 'statName', 'statValue'])

--- 4. FBS Teams Keys ---
dict_keys(['id', 'school', 'mascot', 'abbreviation', 'alternateNames', 'conference', 'classification', 'color', 'alternateColor', 'logos', 'twitter', 'location', 'division'])

--- 5. Roster Keys ---
dict_keys(['id', 'firstName', 'lastName', 'team', 'height', 'weight', 'jersey', 'year', 'position', 'homeCity', 'homeState', 'homeCountry', 'homeLatitude', 'homeLongitude', 'homeCountyFIPS', 'recruitIds'])


In [30]:
# Set up output directory
OUTPUT_DIR = PROJECT_ROOT / "data" / "college_stats" / "cfbd"
if not OUTPUT_DIR.exists():
    print(f"Creating output directory: {OUTPUT_DIR}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Define the timeframe
YEARS = range(2015, 2026) # 2015 through 2025

def fetch_and_save():
    # Initialize empty lists to hold the yearly data chunks
    all_advanced_stats = []
    all_player_stats = []
    all_team_stats = []
    all_fbs_teams = []
    all_rosters = []

    with cfbd.ApiClient(configuration) as api_client:
        stats_api = cfbd.StatsApi(api_client)
        teams_api = cfbd.TeamsApi(api_client)

        for year in YEARS:
            print(f"🏈 Fetching data for {year}...", end=" ")
            
            try:
                # 1. Advanced Season Stats
                adv_stats = stats_api.get_advanced_season_stats(year=year)
                if adv_stats:
                    all_advanced_stats.extend([s.to_dict() for s in adv_stats])

                # 2. Player Season Stats
                # No category filter means we get passing, rushing, receiving, AND defense
                player_stats = stats_api.get_player_season_stats(year=year)
                if player_stats:
                    all_player_stats.extend([s.to_dict() for s in player_stats])

                # 3. Team Stats
                team_stats = stats_api.get_team_stats(year=year)
                if team_stats:
                    all_team_stats.extend([s.to_dict() for s in team_stats])

                # 4. FBS Teams
                fbs_teams = teams_api.get_fbs_teams(year=year)
                if fbs_teams:
                    # Inject the year so we have historical tracking of team alignments
                    team_dicts = [t.to_dict() for t in fbs_teams]
                    for t in team_dicts:
                        t['season'] = year # Use 'season' to stay consistent with other stats
                    all_fbs_teams.extend(team_dicts)

                # 5. Rosters
                rosters = teams_api.get_roster(year=year)
                if rosters:
                    # Inject the year into the roster data for tracking
                    roster_dicts = [r.to_dict() for r in rosters]
                    for r in roster_dicts:
                        r['season_roster'] = year # Use specific key to avoid confusion with player 'year' (FR, SO, etc.)
                    all_rosters.extend(roster_dicts)
                
                print("✅")

            except ApiException as e:
                print(f"⚠️ API Exception in {year}: {e}")
            
            # Be a good citizen to the free API to prevent rate limit timeouts
            time.sleep(1.0)

    print("\n✅ Data fetching complete. Processing and saving to CSVs...")

    # --- Flatten and Save Data ---
    
    # 1. Advanced Team Stats (Requires json_normalize to flatten nested 'offense'/'defense' objects)
    if all_advanced_stats:
        df_adv = pd.json_normalize(all_advanced_stats)
        df_adv.to_csv(OUTPUT_DIR / "advanced_season_stats_2015_2025.csv", index=False)
        print(f"💾 Saved Advanced Season Stats: {df_adv.shape[0]} rows")

    # 2. Player Stats (Wide Format Pivot)
    if all_player_stats:
        df_players_raw = pd.DataFrame(all_player_stats)
        
        # FIX: The dictionary keys from the CFBD library's to_dict() use camelCase
        pivot_index = ['season', 'playerId', 'player', 'team', 'conference', 'category']
        pivot_columns = 'statType'
        pivot_values = 'stat'
        
        # Filter raw data to ensure only rows with the necessary keys are processed
        df_players_wide = df_players_raw.pivot_table(
            index=pivot_index,
            columns=pivot_columns,
            values=pivot_values,
            aggfunc='first'
        ).reset_index()
        df_players_wide.columns.name = None
        
        df_players_wide.to_csv(OUTPUT_DIR / "player_season_stats_2015_2025.csv", index=False)
        print(f"💾 Saved Player Season Stats: {df_players_wide.shape[0]} rows")

    # 3. Traditional Team Stats (Wide Format Pivot)
    if all_team_stats:
        df_teams_raw = pd.DataFrame(all_team_stats)
        
        # FIX: Correct keys are statName and statValue
        df_teams_wide = df_teams_raw.pivot_table(
            index=['season', 'team', 'conference'],
            columns='statName',
            values='statValue',
            aggfunc='first'
        ).reset_index()
        df_teams_wide.columns.name = None
        
        df_teams_wide.to_csv(OUTPUT_DIR / "team_stats_2015_2025.csv", index=False)
        print(f"💾 Saved Traditional Team Stats: {df_teams_wide.shape[0]} rows")

    # 4. FBS Teams List
    if all_fbs_teams:
        df_fbs = pd.json_normalize(all_fbs_teams) # Flattens nested 'location' objects
        df_fbs.to_csv(OUTPUT_DIR / "fbs_teams_2015_2025.csv", index=False)
        print(f"💾 Saved FBS Teams Roster: {df_fbs.shape[0]} rows")

    # 5. Master Player Rosters
    if all_rosters:
        df_rosters = pd.DataFrame(all_rosters)
        df_rosters.to_csv(OUTPUT_DIR / "rosters_2015_2025.csv", index=False)
        print(f"💾 Saved Master Rosters: {df_rosters.shape[0]} rows")

    print(f"\n🎉 All extractions saved to ./{OUTPUT_DIR.name}/")

if __name__ == "__main__":
    fetch_and_save()


🏈 Fetching data for 2015... ✅
🏈 Fetching data for 2016... ✅
🏈 Fetching data for 2017... ✅
🏈 Fetching data for 2018... ✅
🏈 Fetching data for 2019... ✅
🏈 Fetching data for 2020... ✅
🏈 Fetching data for 2021... ✅
🏈 Fetching data for 2022... ✅
🏈 Fetching data for 2023... ✅
🏈 Fetching data for 2024... ✅
🏈 Fetching data for 2025... ✅

✅ Data fetching complete. Processing and saving to CSVs...
💾 Saved Advanced Season Stats: 1475 rows
💾 Saved Player Season Stats: 211284 rows
💾 Saved Traditional Team Stats: 1438 rows
💾 Saved FBS Teams Roster: 1440 rows
💾 Saved Master Rosters: 230008 rows

🎉 All extractions saved to ./cfbd/


In [34]:
import pandas as pd
import csv

# Path to the raw roster file
roster_path = PROJECT_ROOT / "data" / "college_stats" / "cfbd" / "rosters_2015_2025.csv"

if roster_path.exists():
    print(f"📄 Loading roster: {roster_path}")
    df = pd.read_csv(roster_path)
    
    # 1. Clean the 'id' column: Remove the leading '-' and ensure it's numeric
    if 'id' in df.columns:
        # First ensure it's numeric, then take absolute value to remove leading '-'
        df['id'] = pd.to_numeric(df['id'], errors='coerce').fillna(0).astype('int64').abs()
    
    # 2. Final bit of cleaning: Remove any actual double quotes if they exist within string columns
    # (Sometimes CSV crawlers or certain editors wrap strings in extra quotes)
    for col in df.select_dtypes(include=['object']):
        df[col] = df[col].astype(str).str.replace('"', '', regex=False)

    # Save the cleaned file
    output_path = roster_path.parent / "rosters_2015_2025.csv"
    
    # We use QUOTE_MINIMAL (default) instead of QUOTE_ALL to avoid redundant double quotes 
    # around numeric IDs and standard text.
    df.to_csv(
        output_path, 
        index=False, 
        encoding='utf-8-sig',      # Excel friendly
        quoting=csv.QUOTE_MINIMAL, # Only quote when necessary (e.g. contains comma)
        lineterminator='\n'        # Standard line endings
    )
    
    print(f"✅ Success! Cleaned roster saved to: {output_path}")
    display(df.head())
else:
    print(f"❌ File not found at: {roster_path}")

📄 Loading roster: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\college_stats\cfbd\rosters_2015_2025.csv
✅ Success! Cleaned roster saved to: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\college_stats\cfbd\rosters_2015_2025.csv


,id,firstName,lastName,team,height,weight,jersey,year,position,homeCity,homeState,homeCountry,homeLatitude,homeLongitude,homeCountyFIPS,recruitIds,season_roster
0,1044179,Aaron,Young,Wyoming,74.0,198.0,12.0,2015,QB,Sacramento,CA,USA,38.581572,-121.494400,6067.0,['186598'],2015
1,1044178,Eddie,Yarbrough,Wyoming,75.0,251.0,55.0,2015,DE,Aurora,CO,USA,39.729432,-104.831920,8005.0,['154200'],2015
2,1044177,Will,Tutein,Wyoming,74.0,212.0,36.0,2015,LB,St. Croix,nan,U.S.,NaN,NaN,NaN,[],2015
3,1044176,Riley,Stringer,Wyoming,73.0,270.0,76.0,2015,NT,Powell,WY,USA,44.752873,-108.759020,56029.0,[],2015
4,1044175,Ryley,Southam,Wyoming,75.0,279.0,79.0,2015,C,Durham,CA,USA,39.646063,-121.800123,6007.0,[],2015


In [11]:
with cfbd.ApiClient(configuration) as api_client:
    stats_api = cfbd.StatsApi(api_client)
    
    # Set your target team and year
    target_team = "Texas"
    target_year = 2024
    
    print(f"Fetching all player stats for {target_team} in {target_year}...\n")
    
    try:
        # Notice we removed the 'category' filter so we get Offense, Defense, and Special Teams
        player_stats = stats_api.get_player_season_stats(year=target_year, team=target_team)
        
        data = []
        for stat in player_stats:
            data.append({
                'player': stat.player,
                'category': stat.category,
                'stat_type': stat.stat_type,
                'stat_value': stat.stat
            })
            
        df_all_stats = pd.DataFrame(data)
        
        # ---------------------------------------------------------
        # 1. Print a dictionary of all available stat categories
        # ---------------------------------------------------------
        print("--- AVAILABLE STAT CATEGORIES & KEYS ---")
        category_mapping = df_all_stats.groupby('category')['stat_type'].unique().to_dict()
        for cat, types in category_mapping.items():
            print(f"🏈 {cat.upper()}: {', '.join(types)}")
            
        print("\n" + "="*60 + "\n")
        
        # ---------------------------------------------------------
        # 2. Show a specific player's data (Wide Format for XGBoost)
        # ---------------------------------------------------------
        print("--- SAMPLE PLAYER DATA PROFILES ---")
        
        # Make sure values are numeric for sorting
        df_all_stats['stat_value_numeric'] = pd.to_numeric(df_all_stats['stat_value'], errors='coerce')
        
        # A. Find the QB (Most Passing Yards)
        qb_df = df_all_stats[(df_all_stats['category'] == 'passing') & (df_all_stats['stat_type'] == 'YDS')]
        if not qb_df.empty:
            qb_name = qb_df.sort_values(by='stat_value_numeric', ascending=False).iloc[0]['player']
            print(f"\nExample Offensive Profile: {qb_name}")
            
            # Pivot just this player's stats
            qb_profile = df_all_stats[df_all_stats['player'] == qb_name].pivot_table(
                index='player', columns='stat_type', values='stat_value_numeric', aggfunc='first'
            )
            qb_profile.columns.name = None
            print(qb_profile.to_string())

        # B. Find a Top Defender (Most Total Tackles)
        def_df = df_all_stats[(df_all_stats['category'] == 'defensive') & (df_all_stats['stat_type'] == 'TOT')]
        if not def_df.empty:
            def_name = def_df.sort_values(by='stat_value_numeric', ascending=False).iloc[0]['player']
            print(f"\nExample Defensive Profile: {def_name}")
            
            # Pivot just this player's stats
            def_profile = df_all_stats[df_all_stats['player'] == def_name].pivot_table(
                index='player', columns='stat_type', values='stat_value_numeric', aggfunc='first'
            )
            def_profile.columns.name = None
            print(def_profile.to_string())

    except ApiException as e:
        print("Exception when calling StatsApi->get_player_season_stats: %s\n" % e)

Fetching all player stats for Texas in 2024...

--- AVAILABLE STAT CATEGORIES & KEYS ---
🏈 DEFENSIVE: PD, QB HUR, SACKS, SOLO, TD, TFL, TOT
🏈 FUMBLES: FUM, LOST, REC
🏈 INTERCEPTIONS: AVG, INT, TD, YDS
🏈 KICKRETURNS: AVG, LONG, NO, TD, YDS
🏈 KICKING: FGA, FGM, LONG, PCT, PTS, XPA, XPM
🏈 PASSING: ATT, COMPLETIONS, INT, PCT, TD, YDS, YPA
🏈 PUNTRETURNS: AVG, LONG, NO, TD, YDS
🏈 PUNTING: In 20, LONG, NO, TB, YDS, YPP
🏈 RECEIVING: LONG, REC, TD, YDS, YPR
🏈 RUSHING: CAR, LONG, TD, YDS, YPC


--- SAMPLE PLAYER DATA PROFILES ---

Example Offensive Profile: Quinn Ewers
               ATT   CAR  COMPLETIONS  FUM   INT  LONG  LOST    PCT  REC    TD     YDS  YPA  YPC
player                                                                                          
Quinn Ewers  445.0  57.0        293.0  6.0  12.0  26.0   5.0  0.658  0.0  31.0  3472.0  7.8 -1.4

Example Defensive Profile: Anthony Hill Jr.
                   AVG  FUM  INT  LOST   PD  QB HUR  REC  SACKS  SOLO   TD   TFL    TOT   YDS
play

In [ ]:
with cfbd.ApiClient(configuration) as api_client:
    metrics_api = cfbd.MetricsApi(api_client)
        # ---------------------------------------------------------
        # 1. Fetch Player PPA (Predicted Points Added)
        # ---------------------------------------------------------
    try:
        ppa_stats = metrics_api.get_predicted_points_added_by_player_season(
            year=2024, team="Texas", exclude_garbage_time=True)
        print(f"Successfully fetched PPA stats for {len(ppa_stats)} players.")
        print(f"Columns available in PPA stats: {ppa_stats[0].to_dict().keys() if ppa_stats else 'No data'}")
    except ApiException as e:
        print("Exception when calling MetricsApi->get_predicted_points_added_by_player_season: %s\n" % e)

In [ ]:
with cfbd.ApiClient(configuration) as api_client:
    metrics_api = cfbd.MetricsApi(api_client)
    players_api = cfbd.PlayersApi(api_client)
    
    target_team = "Texas"
    target_year = 2024
    
    print(f"Fetching Advanced Player Metrics (PPA & Usage) for {target_team} in {target_year}...\n")
    
    try:
        # ---------------------------------------------------------
        # 1. Fetch Player PPA (Predicted Points Added)
        # ---------------------------------------------------------
        ppa_stats = metrics_api.get_predicted_points_added_by_player_season(
            year=target_year, team=target_team, exclude_garbage_time=True
        )
        
        ppa_data = []
        for stat in ppa_stats:
            avg_ppa = stat.average_ppa
            tot_ppa = stat.total_ppa
            
            # Safely grab the play count regardless of what the python wrapper named it
            play_count = getattr(stat, 'countable_plays', getattr(stat, 'plays', 0))
            
            if avg_ppa and tot_ppa:
                ppa_data.append({
                    'player': stat.name,
                    'position': stat.position,
                    'countable_plays': play_count,
                    
                    # Average PPA per play
                    'avg_ppa_overall': getattr(avg_ppa, 'all', None),
                    'avg_ppa_pass': getattr(avg_ppa, 'pass_', getattr(avg_ppa, 'pass', None)), 
                    'avg_ppa_rush': getattr(avg_ppa, 'rush', None),
                    'avg_ppa_3rd_down': getattr(avg_ppa, 'third_down', None),
                    'avg_ppa_passing_downs': getattr(avg_ppa, 'passing_downs', None),
                    
                    # Total Cumulative PPA
                    'total_ppa_overall': getattr(tot_ppa, 'all', None)
                })
                
        df_ppa = pd.DataFrame(ppa_data)
        
        # ---------------------------------------------------------
        # 2. Fetch Player Usage
        # ---------------------------------------------------------
        usage_stats = players_api.get_player_usage(
            year=target_year, team=target_team, exclude_garbage_time=True
        )
        
        usage_data = []
        # FIX: Now correctly iterating over 'usage_stats' instead of 'usage_data_raw'
        for u in usage_stats:
            if getattr(u, 'usage', None):
                usage_data.append({
                    'player': u.name,
                    'usage_overall': getattr(u.usage, 'overall', None),
                    'usage_pass': getattr(u.usage, 'pass_', getattr(u.usage, 'pass', None)),
                    'usage_rush': getattr(u.usage, 'rush', None),
                    'usage_1st_down': getattr(u.usage, 'first_down', None),
                    'usage_3rd_down': getattr(u.usage, 'third_down', None)
                })
                
        df_usage = pd.DataFrame(usage_data)

        # ---------------------------------------------------------
        # 3. Merge and Display the Advanced Profiles
        # ---------------------------------------------------------
        if not df_ppa.empty and not df_usage.empty:
            # Merge on player name
            df_advanced = pd.merge(df_ppa, df_usage, on='player', how='inner')
            
            print("--- AVAILABLE ADVANCED METRICS ---")
            print("PPA (Predicted Points Added) & Usage by Down/Distance/Play Type")
            print(f"Total Columns: {list(df_advanced.columns)}\n")
            
            print("--- TOP OFFENSIVE CONTRIBUTORS (Ranked by Total Expected Points Added) ---")
            
            # Sort by total value generated for the team
            df_advanced_sorted = df_advanced.sort_values(by='total_ppa_overall', ascending=False)
            
            # Displaying a curated subset of the most valuable ML features
            display_cols = [
                'player', 'position', 'countable_plays', 'usage_overall', 
                'avg_ppa_overall', 'avg_ppa_3rd_down', 'total_ppa_overall'
            ]
            display(df_advanced_sorted[display_cols].head(5))

    except ApiException as e:
        print("Exception when fetching Advanced Player Metrics: %s\n" % e)

Fetching Advanced Player Metrics (PPA & Usage) for Texas in 2024...

--- AVAILABLE ADVANCED METRICS ---
PPA (Predicted Points Added) & Usage by Down/Distance/Play Type
Total Columns: ['player', 'position', 'countable_plays', 'avg_ppa_overall', 'avg_ppa_pass', 'avg_ppa_rush', 'avg_ppa_3rd_down', 'avg_ppa_passing_downs', 'total_ppa_overall', 'usage_overall', 'usage_pass', 'usage_rush', 'usage_1st_down', 'usage_3rd_down']

--- TOP OFFENSIVE CONTRIBUTORS (Ranked by Total Expected Points Added) ---


,player,position,countable_plays,usage_overall,avg_ppa_overall,avg_ppa_3rd_down,total_ppa_overall
11,Quinn Ewers,QB,0,0.554,0.321,0.603,156.045
4,Matthew Golden,WR,0,0.074,1.055,1.974,77.045
3,Gunnar Helm,TE,0,0.070,0.910,1.306,62.780
9,Arch Manning,QB,0,0.171,0.712,1.551,61.978
5,Isaiah Bond,WR,0,0.061,1.053,1.023,50.542



--- POSITIONAL PPA EFFICIENCY ---
Quarterback Pass Efficiency vs Running Back Rush Efficiency:


TypeError: unsupported format string passed to NoneType.__format__

In [5]:
with cfbd.ApiClient(configuration) as api_client:
    games_api = cfbd.GamesApi(api_client)
    
    # We will look at Texas vs Colorado State (Week 1, 2024)
    print("🔍 Fetching Game Player Stats for Texas (2024, Week 1)...")
    
    try:
        # Pulling the full box score for the game
        game_stats = games_api.get_game_player_stats(year=2024, week=1, team="Texas")
        
        records = []
        for game in game_stats:
            for team_obj in game.teams:
                # FIX: Accessing .team instead of .school
                if team_obj.team == "Texas":
                    for category in team_obj.categories:
                        for stat_type in category.types:
                            for athlete in stat_type.athletes:
                                records.append({
                                    'player_name': athlete.name,
                                    'player_id': athlete.id,
                                    'category': category.name,
                                    'stat_type': stat_type.name,
                                    'stat_value': athlete.stat
                                })
        
        df_game = pd.DataFrame(records)
        
        # 1. Inspect all unique categories returned in the box score
        print("\n--- STAT CATEGORIES FOUND ---")
        print(df_game['category'].unique())
        
        # 2. SEARCH FOR OFFENSIVE LINEMEN
        # Known Texas OL: Kelvin Banks Jr., Hayden Conner, Jake Majors, DJ Campbell, Cameron Williams
        ol_names = ["Kelvin Banks", "Hayden Conner", "Jake Majors", "DJ Campbell", "Cameron Williams"]
        
        print("\n--- TRENCH PLAYER SEARCH ---")
        trench_results = df_game[df_game['player_name'].str.contains('|'.join(ol_names), case=False, na=False)]
        
        if trench_results.empty:
            print("❌ No Offensive Linemen found in the box score.")
            print("Note: If an OL didn't catch a pass or recover a fumble, they won't appear here.")
        else:
            print("✅ Found data for the following OL:")
            print(trench_results[['player_name', 'category', 'stat_type', 'stat_value']])

    except ApiException as e:
        print("Exception when calling GamesApi->get_game_player_stats: %s\n" % e)

🔍 Fetching Game Player Stats for Texas (2024, Week 1)...

--- STAT CATEGORIES FOUND ---
['passing' 'rushing' 'receiving' 'defensive' 'interceptions' 'kickReturns'
 'puntReturns' 'kicking' 'punting' 'fumbles']

--- TRENCH PLAYER SEARCH ---
✅ Found data for the following OL:
     player_name   category stat_type stat_value
763  DJ Campbell  defensive       TFL          0
789  DJ Campbell  defensive        PD          0
817  DJ Campbell  defensive    QB HUR          0
845  DJ Campbell  defensive        TD          0
873  DJ Campbell  defensive       TOT          1
901  DJ Campbell  defensive      SOLO          1
929  DJ Campbell  defensive     SACKS          0


In [19]:
import os
import time
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.edge.service import Service as EdgeService
from selenium.common.exceptions import SessionNotCreatedException

# --- CONFIGURATION ---
# Using your X: drive path for Edge driver
DRIVER_PATH = r"X:\Python\browsers\edge_driver\msedgedriver.exe"
PFF_DOWNLOAD_DIR = PROJECT_ROOT / "data" / "college_stats" / "pff_csvs"
PFF_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

YEARS = range(2015, 2026)
PHASES = [
    ("offense-blocking", "offense"),
    ("defense", "defense")
]

def setup_driver():
    options = webdriver.EdgeOptions()
    # PFF blocks traditional headless; keep this windowed for the login phase
    prefs = {
        "download.default_directory": str(PFF_DOWNLOAD_DIR.absolute()),
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True
    }
    options.add_experimental_option("prefs", prefs)
    
    if Path(DRIVER_PATH).exists():
        try:
            service = EdgeService(executable_path=DRIVER_PATH)
            return webdriver.Edge(service=service, options=options)
        except SessionNotCreatedException as e:
            print(f"⚠️ Local Edge driver mismatch ({e.msg.splitlines()[0]}). Falling back to Selenium Manager...")

    print("ℹ️ Starting Edge with Selenium Manager (auto-matching driver)...")
    return webdriver.Edge(options=options)

def wait_for_download_and_rename(target_filename, files_before_click):
    """
    Watches the folder for a new .csv, waits for completion, 
    and renames it to the target_filename.
    """
    print(f"   ⏳ Waiting for download to complete...")
    timeout = 90
    seconds = 0
    
    while seconds < timeout:
        # Look for Edge/Chromium temporary download files
        dl_files = list(PFF_DOWNLOAD_DIR.glob("*.crdownload"))
        current_files = {p.resolve() for p in PFF_DOWNLOAD_DIR.glob("*.csv")}
        new_files = [Path(p) for p in current_files if p not in files_before_click]

        if not dl_files and new_files:
            candidate = max(new_files, key=lambda p: p.stat().st_mtime)
            dest = PFF_DOWNLOAD_DIR / f"{target_filename}.csv"
            # If the file already exists (from a previous run), remove it
            if dest.exists():
                dest.unlink()
            if candidate.resolve() != dest.resolve():
                candidate.rename(dest)
            print(f"   ✅ Successfully saved as: {target_filename}.csv")
            return True
        
        time.sleep(1)
        seconds += 1
    
    print(f"   ❌ Download timed out for {target_filename}")
    return False

def download_pff_csvs():
    driver = setup_driver()
    wait = WebDriverWait(driver, 20)
    
    try:
        # 1. INITIAL LOGIN
        # Open a known page; if your session is active, downloads can proceed immediately
        driver.get("https://premium.pff.com/ncaa/positions/2024/REG/defense?division=fbs")
        print("ℹ️ Opened PFF landing page. Continuing without fixed login delay...")

        for year in YEARS:
            for pff_endpoint, label in PHASES:
                url = f"https://premium.pff.com/ncaa/positions/{year}/REG/{pff_endpoint}?division=fbs"
                filename = f"pff_{label}_{year}"
                
                # Skip if we already have this file cleaned and named
                if (PFF_DOWNLOAD_DIR / f"{filename}.csv").exists():
                    print(f"⏭️  Skipping {year} {label} (File already exists).")
                    continue

                print(f"🏈 Navigating to {label.upper()} stats for {year}...")
                driver.get(url)
                
                try:
                    # 2. FIND CSV BUTTON
                    csv_button = wait.until(EC.element_to_be_clickable(
                        (By.XPATH, "//button[contains(., 'CSV')]")
                    ))
                    
                    # 3. TRIGGER DOWNLOAD
                    print(f"   -> Clicking CSV Export...")
                    files_before_click = {p.resolve() for p in PFF_DOWNLOAD_DIR.glob("*.csv")}
                    csv_button.click()
                    
                    # 4. WATCH AND RENAME
                    wait_for_download_and_rename(filename, files_before_click)
                    
                except Exception as e:
                    print(f"   ⚠️ Error processing {year} {label}: {str(e)[:50]}...")
                    continue

        print("\n🎉 Bulk download and renaming complete.")

    finally:
        time.sleep(5)
        driver.quit()

if __name__ == "__main__":
    download_pff_csvs()

⚠️ Local Edge driver mismatch (session not created: This version of Microsoft Edge WebDriver only supports Microsoft Edge version 143). Falling back to Selenium Manager...
ℹ️ Starting Edge with Selenium Manager (auto-matching driver)...
ℹ️ Opened PFF landing page. Continuing without fixed login delay...
🏈 Navigating to OFFENSE stats for 2015...
   -> Clicking CSV Export...
   ⏳ Waiting for download to complete...
   ✅ Successfully saved as: pff_offense_2015.csv
🏈 Navigating to DEFENSE stats for 2015...
   -> Clicking CSV Export...
   ⏳ Waiting for download to complete...
   ✅ Successfully saved as: pff_defense_2015.csv
🏈 Navigating to OFFENSE stats for 2016...
   -> Clicking CSV Export...
   ⏳ Waiting for download to complete...
   ✅ Successfully saved as: pff_offense_2016.csv
🏈 Navigating to DEFENSE stats for 2016...
   -> Clicking CSV Export...
   ⏳ Waiting for download to complete...
   ✅ Successfully saved as: pff_defense_2016.csv
🏈 Navigating to OFFENSE stats for 2017...
   -> Cli